<a href="https://colab.research.google.com/github/SrustiUB/Data-Science/blob/main/Time_series_Momentum.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# ============================================================
# TIME-SERIES MOMENTUM (TSMOM)
# NIFTY 50 ALGO TRADING STRATEGY
#
# Single instrument
# Long / Short
# backtesting.py
# QuantStats
# ============================================================


# ============================================================
# 1. INSTALL
# ============================================================

!pip install -q yfinance backtesting quantstats


# ============================================================
# 2. IMPORTS
# ============================================================

import yfinance as yf
import pandas as pd
import numpy as np
import quantstats as qs

from backtesting import Backtest, Strategy
import backtesting

backtesting.set_bokeh_output(notebook=False)


# ============================================================
# 3. DOWNLOAD NIFTY 50
# ============================================================

data = yf.download(
    "^NSEI",
    start="2018-01-01",
    end="2026-01-01",
    auto_adjust=True,
    progress=False
)

# Handle yfinance MultiIndex
if isinstance(data.columns, pd.MultiIndex):
    data.columns = data.columns.get_level_values(0)

data = data[
    ["Open", "High", "Low", "Close", "Volume"]
].dropna()

print("Data shape:", data.shape)
print(data.head())


# ============================================================
# 4. TIME-SERIES MOMENTUM
# ============================================================

def ts_momentum(close, window=60):

    close = np.asarray(
        close,
        dtype=float
    )

    signal = np.full(
        len(close),
        np.nan
    )

    signal[window:] = (
        close[window:] /
        close[:-window]
    ) - 1

    return signal


# ============================================================
# 5. STRATEGY
# ============================================================

class TSMOMStrategy(Strategy):

    # Momentum lookback
    lookback = 60

    # Entry threshold
    threshold = 0.00

    def init(self):

        self.signal = self.I(
            ts_momentum,
            self.data.Close,
            self.lookback,
            name="TS Momentum"
        )

    def next(self):

        signal = self.signal[-1]

        if np.isnan(signal):
            return


        # ====================================================
        # LONG
        # ====================================================

        if signal > self.threshold:

            # Close short
            if self.position.is_short:
                self.position.close()

            # Enter long
            if not self.position:
                self.buy()


        # ====================================================
        # SHORT
        # ====================================================

        elif signal < -self.threshold:

            # Close long
            if self.position.is_long:
                self.position.close()

            # Enter short
            if not self.position:
                self.sell()


        # ====================================================
        # FLAT
        # ====================================================

        else:

            if self.position:
                self.position.close()


# ============================================================
# 6. CREATE BACKTEST
# ============================================================

bt = Backtest(
    data,
    TSMOMStrategy,

    cash=100000,

    # 0.05% transaction cost
    commission=0.0005,

    exclusive_orders=True,

    # Close open trades at end
    finalize_trades=True
)


# ============================================================
# 7. RUN
# ============================================================

stats = bt.run()


# ============================================================
# 8. RESULTS
# ============================================================

print("\n")
print("=" * 65)
print("NIFTY 50 TIME-SERIES MOMENTUM")
print("=" * 65)

print(
    f"Return:              "
    f"{stats['Return [%]']:.2f}%"
)

print(
    f"Buy & Hold Return:   "
    f"{stats['Buy & Hold Return [%]']:.2f}%"
)

print(
    f"CAGR:                "
    f"{stats['CAGR [%]']:.2f}%"
)

print(
    f"Annual Volatility:   "
    f"{stats['Volatility (Ann.) [%]']:.2f}%"
)

print(
    f"Sharpe Ratio:        "
    f"{stats['Sharpe Ratio']:.2f}"
)

print(
    f"Sortino Ratio:       "
    f"{stats['Sortino Ratio']:.2f}"
)

print(
    f"Max Drawdown:        "
    f"{stats['Max. Drawdown [%]']:.2f}%"
)

print(
    f"Win Rate:            "
    f"{stats['Win Rate [%]']:.2f}%"
)

print(
    f"Number of Trades:    "
    f"{stats['# Trades']}"
)

print(
    f"Exposure Time:       "
    f"{stats['Exposure Time [%]']:.2f}%"
)

print(
    f"Profit Factor:       "
    f"{stats['Profit Factor']:.2f}"
)


# ============================================================
# 9. BACKTEST CHART
# ============================================================

bt.plot(
    filename="NIFTY_TSMOM_Backtest.html",
    open_browser=False
)

print(
    "\nBacktest chart saved:"
)

print(
    "NIFTY_TSMOM_Backtest.html"
)


# ============================================================
# 10. EQUITY CURVE
# ============================================================

equity = stats[
    "_equity_curve"
]["Equity"]

returns = (
    equity
    .pct_change()
    .dropna()
)


# ============================================================
# 11. QUANTSTATS
# ============================================================

print("\n")
print("=" * 65)
print("QUANTSTATS")
print("=" * 65)

print(
    "CAGR:",
    f"{qs.stats.cagr(returns) * 100:.2f}%"
)

print(
    "Sharpe:",
    f"{qs.stats.sharpe(returns):.2f}"
)

print(
    "Sortino:",
    f"{qs.stats.sortino(returns):.2f}"
)

print(
    "Max Drawdown:",
    f"{qs.stats.max_drawdown(returns) * 100:.2f}%"
)

print(
    "Calmar:",
    f"{qs.stats.calmar(returns):.2f}"
)


# ============================================================
# 12. FULL QUANTSTATS REPORT
# ============================================================

qs.reports.metrics(
    returns,
    mode="full"
)


# ============================================================
# 13. HTML REPORT
# ============================================================

qs.reports.html(
    returns,
    output="NIFTY_TSMOM_QuantStats.html",
    title="NIFTY 50 Time-Series Momentum"
)

print(
    "\nQuantStats report saved:"
)

print(
    "NIFTY_TSMOM_QuantStats.html"
)


# ============================================================
# 14. PARAMETER OPTIMIZATION
# ============================================================

print("\n")
print("=" * 65)
print("PARAMETER OPTIMIZATION")
print("=" * 65)

optimized_stats = bt.optimize(

    lookback=[
        20,
        30,
        40,
        50,
        60,
        90,
        120,
        180
    ],

    threshold=[
        0.00,
        0.01,
        0.02,
        0.03,
        0.05
    ],

    maximize="Sharpe Ratio",

    return_heatmap=False,

    max_tries=40,

    random_state=42
)


# ============================================================
# 15. BEST PARAMETERS
# ============================================================

best_strategy = (
    optimized_stats["_strategy"]
)

best_lookback = (
    best_strategy.lookback
)

best_threshold = (
    best_strategy.threshold
)

print("\n")
print("=" * 65)
print("BEST PARAMETERS")
print("=" * 65)

print(
    "Lookback:",
    best_lookback
)

print(
    "Threshold:",
    best_threshold
)


# ============================================================
# 16. OPTIMIZED PERFORMANCE
# ============================================================

print("\n")
print("=" * 65)
print("OPTIMIZED PERFORMANCE")
print("=" * 65)

print(
    f"Return:              "
    f"{optimized_stats['Return [%]']:.2f}%"
)

print(
    f"CAGR:                "
    f"{optimized_stats['CAGR [%]']:.2f}%"
)

print(
    f"Sharpe Ratio:        "
    f"{optimized_stats['Sharpe Ratio']:.2f}"
)

print(
    f"Sortino Ratio:       "
    f"{optimized_stats['Sortino Ratio']:.2f}"
)

print(
    f"Max Drawdown:        "
    f"{optimized_stats['Max. Drawdown [%]']:.2f}%"
)

print(
    f"Win Rate:            "
    f"{optimized_stats['Win Rate [%]']:.2f}%"
)

print(
    f"Trades:              "
    f"{optimized_stats['# Trades']}"
)


# ============================================================
# END
# ============================================================

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 190.8/190.8 kB 6.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 90.1/90.1 kB 4.5 MB/s eta 0:00:00


/usr/local/lib/python3.12/dist-packages/backtesting/_plotting.py:55: UserWarning: Jupyter Notebook detected. Setting Bokeh output to notebook. This may not work in Jupyter clients without JavaScript support, such as old IDEs. Reset with `backtesting.set_bokeh_output(notebook=False)`.
  warnings.warn('Jupyter Notebook detected. '


Data shape: (1972, 5)
Price               Open          High           Low         Close  Volume
Date                                                                      
2018-01-02  10477.549805  10495.200195  10404.650391  10442.200195  153400
2018-01-03  10482.650391  10503.599609  10429.549805  10443.200195  167300
2018-01-04  10469.400391  10513.000000  10441.450195  10504.799805  174900
2018-01-05  10534.250000  10566.099609  10520.099609  10558.849609  180900
2018-01-08  10591.700195  10631.200195  10588.549805  10623.599609  169000


Backtest.run:   0%|          | 0/1911 [00:00<?, ?bar/s]



NIFTY 50 TIME-SERIES MOMENTUM
Return:              63.45%
Buy & Hold Return:   155.05%
CAGR:                6.34%
Annual Volatility:   14.92%
Sharpe Ratio:        0.43
Sortino Ratio:       0.68
Max Drawdown:        -24.42%
Win Rate:            39.68%
Number of Trades:    63
Exposure Time:       96.86%
Profit Factor:       1.86

Backtest chart saved:
NIFTY_TSMOM_Backtest.html


QUANTSTATS
CAGR: 6.48%
Sharpe: 0.52
Sortino: 0.76
Max Drawdown: -24.42%
Calmar: 0.27

Parameter       Value
--------------  -------
Risk-Free Rate  0.0%
Periods/Year    252
Compounded      Yes
Match Dates     Yes


                           Strategy
-------------------------  ----------
Start Period               2018-01-03
End Period                 2025-12-31
Risk-Free Rate             0.0%
Time in Market             97.0%

Cumulative Return          63.45%
CAGR﹪                     6.48%

Sharpe                     0.52
Prob. Sharpe Ratio         92.8%
Smart Sharpe               0.52
Sortino                


QuantStats report saved:
NIFTY_TSMOM_QuantStats.html


PARAMETER OPTIMIZATION


Backtest.optimize:   0%|          | 0/40 [00:00<?, ?it/s]

Backtest.run:   0%|          | 0/1911 [00:00<?, ?bar/s]



BEST PARAMETERS
Lookback: 60
Threshold: 0.01


OPTIMIZED PERFORMANCE
Return:              92.71%
CAGR:                8.55%
Sharpe Ratio:        0.60
Sortino Ratio:       0.97
Max Drawdown:        -17.14%
Win Rate:            45.07%
Trades:              71


In [ ]:
# ============================================================
# TIME-SERIES MOMENTUM (TSMOM)
# NIFTY 50 ALGO TRADING STRATEGY
#
# 70/30 OUT-OF-SAMPLE + WALK-FORWARD TEST
#
# Single instrument
# Long / Short
# backtesting.py
# QuantStats
# ============================================================


# ============================================================
# 1. INSTALL
# ============================================================

!pip install -q yfinance backtesting quantstats


# ============================================================
# 2. IMPORTS
# ============================================================

import yfinance as yf
import pandas as pd
import numpy as np
import quantstats as qs

from backtesting import Backtest, Strategy
import backtesting

backtesting.set_bokeh_output(notebook=False)


# ============================================================
# 3. DOWNLOAD NIFTY 50
# ============================================================

data = yf.download(
    "^NSEI",
    start="2018-01-01",
    end="2026-01-01",
    auto_adjust=True,
    progress=False
)

if isinstance(data.columns, pd.MultiIndex):
    data.columns = data.columns.get_level_values(0)

data = data[
    ["Open", "High", "Low", "Close", "Volume"]
].dropna()

print("Data shape:", data.shape)
print(data.head())


# ============================================================
# 4. TIME-SERIES MOMENTUM
# ============================================================

def ts_momentum(close, window=60):

    close = np.asarray(
        close,
        dtype=float
    )

    signal = np.full(
        len(close),
        np.nan
    )

    signal[window:] = (
        close[window:] /
        close[:-window]
    ) - 1

    return signal


# ============================================================
# 5. STRATEGY
# ============================================================

class TSMOMStrategy(Strategy):

    lookback = 60
    threshold = 0.01

    def init(self):

        self.signal = self.I(
            ts_momentum,
            self.data.Close,
            self.lookback,
            name="TS Momentum"
        )

    def next(self):

        signal = self.signal[-1]

        if np.isnan(signal):
            return


        # ====================================================
        # LONG
        # ====================================================

        if signal > self.threshold:

            if self.position.is_short:
                self.position.close()

            if not self.position:
                self.buy()


        # ====================================================
        # SHORT
        # ====================================================

        elif signal < -self.threshold:

            if self.position.is_long:
                self.position.close()

            if not self.position:
                self.sell()


        # ====================================================
        # FLAT
        # ====================================================

        else:

            if self.position:
                self.position.close()


# ============================================================
# 6. BACKTEST FUNCTION
# ============================================================

def run_backtest(
    df,
    lookback,
    threshold
):

    bt = Backtest(
        df,
        TSMOMStrategy,
        cash=100000,
        commission=0.0005,
        exclusive_orders=True,
        finalize_trades=True
    )

    stats = bt.run(
        lookback=lookback,
        threshold=threshold
    )

    return stats


# ============================================================
# 7. OPTIMIZATION FUNCTION
# ============================================================

def optimize_strategy(df):

    bt = Backtest(
        df,
        TSMOMStrategy,
        cash=100000,
        commission=0.0005,
        exclusive_orders=True,
        finalize_trades=True
    )


    # --------------------------------------------------------
    # First generate all parameter combinations
    # --------------------------------------------------------

    parameter_combinations = []

    lookbacks = [
        20,
        30,
        40,
        50,
        60,
        90,
        120,
        180
    ]

    thresholds = [
        0.00,
        0.01,
        0.02,
        0.03,
        0.05
    ]


    # --------------------------------------------------------
    # Test each combination
    # --------------------------------------------------------

    best_stats = None
    best_score = -np.inf

    best_lookback = None
    best_threshold = None


    for lookback in lookbacks:

        for threshold in thresholds:

            stats = bt.run(
                lookback=lookback,
                threshold=threshold
            )


            trades = stats["# Trades"]
            sharpe = stats["Sharpe Ratio"]


            # ------------------------------------------------
            # Ignore strategies with too few trades
            # ------------------------------------------------

            if trades < 5:
                continue

            # Ignore invalid Sharpe
            if np.isnan(sharpe):
                continue


            # ------------------------------------------------
            # Score = Sharpe
            # ------------------------------------------------

            score = sharpe


            if score > best_score:

                best_score = score

                best_stats = stats

                best_lookback = lookback

                best_threshold = threshold


    return (
        best_stats,
        best_lookback,
        best_threshold
    )


# ============================================================
# 8. 70/30 OUT-OF-SAMPLE TEST
# ============================================================

print("\n")
print("=" * 70)
print("70/30 OUT-OF-SAMPLE TEST")
print("=" * 70)


split = int(
    len(data) * 0.70
)


train = data.iloc[
    :split
].copy()

test = data.iloc[
    split:
].copy()


print(
    "Training:",
    train.index[0].date(),
    "→",
    train.index[-1].date()
)

print(
    "Testing:",
    test.index[0].date(),
    "→",
    test.index[-1].date()
)


# ============================================================
# 9. OPTIMIZE TRAINING DATA ONLY
# ============================================================

train_stats, best_lookback, best_threshold = (
    optimize_strategy(train)
)


print("\n")
print("=" * 70)
print("TRAINING OPTIMIZATION")
print("=" * 70)

print(
    "Best Lookback:",
    best_lookback
)

print(
    "Best Threshold:",
    best_threshold
)

print(
    "Training Return:",
    f"{train_stats['Return [%]']:.2f}%"
)

print(
    "Training CAGR:",
    f"{train_stats['CAGR [%]']:.2f}%"
)

print(
    "Training Sharpe:",
    f"{train_stats['Sharpe Ratio']:.2f}"
)

print(
    "Training Trades:",
    train_stats["# Trades"]
)


# ============================================================
# 10. TEST ON COMPLETELY UNSEEN DATA
# ============================================================

oos_stats = run_backtest(
    test,
    best_lookback,
    best_threshold
)


print("\n")
print("=" * 70)
print("OUT-OF-SAMPLE RESULTS")
print("=" * 70)

print(
    f"Return:              "
    f"{oos_stats['Return [%]']:.2f}%"
)

print(
    f"CAGR:                "
    f"{oos_stats['CAGR [%]']:.2f}%"
)

print(
    f"Sharpe:              "
    f"{oos_stats['Sharpe Ratio']:.2f}"
)

print(
    f"Sortino:             "
    f"{oos_stats['Sortino Ratio']:.2f}"
)

print(
    f"Max Drawdown:        "
    f"{oos_stats['Max. Drawdown [%]']:.2f}%"
)

print(
    f"Win Rate:            "
    f"{oos_stats['Win Rate [%]']:.2f}%"
)

print(
    f"Trades:              "
    f"{oos_stats['# Trades']}"
)


# ============================================================
# 11. OOS QUANTSTATS
# ============================================================

oos_equity = (
    oos_stats[
        "_equity_curve"
    ]["Equity"]
)

oos_returns = (
    oos_equity
    .pct_change()
    .dropna()
)


print("\n")
print("=" * 70)
print("OOS QUANTSTATS")
print("=" * 70)

print(
    "CAGR:",
    f"{qs.stats.cagr(oos_returns) * 100:.2f}%"
)

print(
    "Sharpe:",
    f"{qs.stats.sharpe(oos_returns):.2f}"
)

print(
    "Sortino:",
    f"{qs.stats.sortino(oos_returns):.2f}"
)

print(
    "Max Drawdown:",
    f"{qs.stats.max_drawdown(oos_returns) * 100:.2f}%"
)

print(
    "Calmar:",
    f"{qs.stats.calmar(oos_returns):.2f}"
)


# ============================================================
# 12. OOS QUANTSTATS REPORT
# ============================================================

qs.reports.html(
    oos_returns,
    output="NIFTY_TSMOM_OOS.html",
    title="NIFTY 50 TSMOM - Out-of-Sample"
)

print(
    "\nOOS QuantStats report:"
)

print(
    "NIFTY_TSMOM_OOS.html"
)


# ============================================================
# 13. WALK-FORWARD TEST
# ============================================================

print("\n")
print("=" * 70)
print("WALK-FORWARD TEST")
print("=" * 70)


# ------------------------------------------------------------
# 2 YEARS TRAINING
# 6 MONTHS TEST
# ------------------------------------------------------------

TRAIN_DAYS = 504
TEST_DAYS = 126


wf_results = []

start = 0


# ============================================================
# 14. WALK-FORWARD LOOP
# ============================================================

while (
    start +
    TRAIN_DAYS +
    TEST_DAYS
    <= len(data)
):


    # ========================================================
    # TRAINING WINDOW
    # ========================================================

    train_start = start

    train_end = (
        start +
        TRAIN_DAYS
    )

    test_end = (
        train_end +
        TEST_DAYS
    )


    train_wf = data.iloc[
        train_start:train_end
    ].copy()


    test_wf = data.iloc[
        train_end:test_end
    ].copy()


    # ========================================================
    # OPTIMIZE ONLY TRAINING WINDOW
    # ========================================================

    (
        wf_train_stats,
        wf_lookback,
        wf_threshold
    ) = optimize_strategy(
        train_wf
    )


    # ========================================================
    # TEST FUTURE WINDOW
    # ========================================================

    wf_test_stats = run_backtest(
        test_wf,
        wf_lookback,
        wf_threshold
    )


    # ========================================================
    # SAVE RESULTS
    # ========================================================

    wf_results.append({

        "Train Start":
            train_wf.index[0],

        "Train End":
            train_wf.index[-1],

        "Test Start":
            test_wf.index[0],

        "Test End":
            test_wf.index[-1],

        "Lookback":
            wf_lookback,

        "Threshold":
            wf_threshold,

        "Return":
            wf_test_stats[
                "Return [%]"
            ],

        "CAGR":
            wf_test_stats[
                "CAGR [%]"
            ],

        "Sharpe":
            wf_test_stats[
                "Sharpe Ratio"
            ],

        "Sortino":
            wf_test_stats[
                "Sortino Ratio"
            ],

        "Max Drawdown":
            wf_test_stats[
                "Max. Drawdown [%]"
            ],

        "Trades":
            wf_test_stats[
                "# Trades"
            ]
    })


    # ========================================================
    # DISPLAY WINDOW
    # ========================================================

    print("\n")
    print(
        f"Window {len(wf_results)}"
    )

    print(
        "Train:",
        train_wf.index[0].date(),
        "→",
        train_wf.index[-1].date()
    )

    print(
        "Test:",
        test_wf.index[0].date(),
        "→",
        test_wf.index[-1].date()
    )

    print(
        "Lookback:",
        wf_lookback
    )

    print(
        "Threshold:",
        wf_threshold
    )

    print(
        "OOS Return:",
        f"{wf_test_stats['Return [%]']:.2f}%"
    )

    print(
        "OOS Sharpe:",
        f"{wf_test_stats['Sharpe Ratio']:.2f}"
    )

    print(
        "Trades:",
        wf_test_stats["# Trades"]
    )


    # Move forward 6 months
    start += TEST_DAYS


# ============================================================
# 15. WALK-FORWARD DATAFRAME
# ============================================================

wf_df = pd.DataFrame(
    wf_results
)


print("\n")
print("=" * 70)
print("WALK-FORWARD RESULTS")
print("=" * 70)

print(
    wf_df.to_string(
        index=False
    )
)


# ============================================================
# 16. WALK-FORWARD SUMMARY
# ============================================================

print("\n")
print("=" * 70)
print("WALK-FORWARD SUMMARY")
print("=" * 70)


# Remove NaN Sharpe values
valid_sharpe = wf_df[
    "Sharpe"
].dropna()


print(
    "Number of OOS windows:",
    len(wf_df)
)

print(
    "Average OOS Return:",
    f"{wf_df['Return'].mean():.2f}%"
)

print(
    "Median OOS Return:",
    f"{wf_df['Return'].median():.2f}%"
)

print(
    "Average OOS CAGR:",
    f"{wf_df['CAGR'].mean():.2f}%"
)

print(
    "Average OOS Sharpe:",
    f"{valid_sharpe.mean():.2f}"
)

print(
    "Median OOS Sharpe:",
    f"{valid_sharpe.median():.2f}"
)

print(
    "Average OOS Sortino:",
    f"{wf_df['Sortino'].mean():.2f}"
)

print(
    "Average OOS Max Drawdown:",
    f"{wf_df['Max Drawdown'].mean():.2f}%"
)

print(
    "Positive OOS Windows:",
    (
        wf_df["Return"] > 0
    ).sum(),
    "/",
    len(wf_df)
)

print(
    "Positive OOS Sharpe:",
    (
        wf_df["Sharpe"] > 0
    ).sum(),
    "/",
    len(wf_df)
)


# ============================================================
# 17. PARAMETER STABILITY
# ============================================================

print("\n")
print("=" * 70)
print("PARAMETER STABILITY")
print("=" * 70)

print(
    wf_df[
        [
            "Test Start",
            "Test End",
            "Lookback",
            "Threshold",
            "Return",
            "Sharpe",
            "Trades"
        ]
    ].to_string(
        index=False
    )
)


# ============================================================
# 18. SAVE WALK-FORWARD RESULTS
# ============================================================

wf_df.to_csv(
    "NIFTY_TSMOM_Walk_Forward.csv",
    index=False
)

print(
    "\nWalk-forward results saved:"
)

print(
    "NIFTY_TSMOM_Walk_Forward.csv"
)


# ============================================================
# 19. FINAL RESEARCH INTERPRETATION
# ============================================================

print("\n")
print("=" * 70)
print("RESEARCH INTERPRETATION")
print("=" * 70)


avg_sharpe = valid_sharpe.mean()

positive_windows = (
    wf_df["Return"] > 0
).sum()

total_windows = len(wf_df)


print(
    f"Average OOS Sharpe: "
    f"{avg_sharpe:.2f}"
)

print(
    f"Positive OOS windows: "
    f"{positive_windows}/{total_windows}"
)


if avg_sharpe > 0:

    print(
        "\nTSMOM shows positive average "
        "out-of-sample risk-adjusted performance."
    )

else:

    print(
        "\nTSMOM does not show positive "
        "average out-of-sample Sharpe."
    )


print(
    "\nParameters were selected using "
    "training data only."
)

print(
    "Test windows were never used "
    "for parameter selection."
)

print(
    "\nEnd of TSMOM walk-forward test."
)

Data shape: (1972, 5)
Price               Open          High           Low         Close  Volume
Date                                                                      
2018-01-02  10477.549805  10495.200195  10404.650391  10442.200195  153400
2018-01-03  10482.650391  10503.599609  10429.549805  10443.200195  167300
2018-01-04  10469.400391  10513.000000  10441.450195  10504.799805  174900
2018-01-05  10534.250000  10566.099609  10520.099609  10558.849609  180900
2018-01-08  10591.700195  10631.200195  10588.549805  10623.599609  169000


70/30 OUT-OF-SAMPLE TEST
Training: 2018-01-02 → 2023-08-07
Testing: 2023-08-08 → 2025-12-31


Backtest.run:   0%|          | 0/1359 [00:00<?, ?bar/s]

Backtest.run:   0%|          | 0/1359 [00:00<?, ?bar/s]

Backtest.run:   0%|          | 0/1359 [00:00<?, ?bar/s]

Backtest.run:   0%|          | 0/1359 [00:00<?, ?bar/s]

...

Backtest.run:   0%|          | 0/323 [00:00<?, ?bar/s]

Backtest.run:   0%|          | 0/323 [00:00<?, ?bar/s]

Backtest.run:   0%|          | 0/323 [00:00<?, ?bar/s]

Backtest.run:   0%|          | 0/65 [00:00<?, ?bar/s]



Window 11
Train: 2023-02-10 → 2025-02-25
Test: 2025-02-27 → 2025-09-02
Lookback: 60
Threshold: 0.0
OOS Return: -4.74%
OOS Sharpe: -1.60
Trades: 6


WALK-FORWARD RESULTS
Train Start  Train End Test Start   Test End  Lookback  Threshold     Return       CAGR    Sharpe   Sortino  Max Drawdown  Trades
 2018-01-02 2020-01-24 2020-01-27 2020-07-30        30       0.03  14.246440  30.076907  0.675288  1.381128    -22.305657       5
 2018-07-05 2020-07-30 2020-07-31 2021-01-29        30       0.03   8.606427  18.020462  1.370120  2.188026     -7.274169       7
 2019-01-11 2021-01-29 2021-02-01 2021-08-04        40       0.01   2.981243   6.004812  0.630223  1.101926     -5.468922       9
 2019-07-19 2021-08-04 2021-08-05 2022-02-04        40       0.00 -13.346512 -24.867633 -2.919758 -2.802480    -15.979750       5
 2020-01-27 2022-02-04 2022-02-07 2022-08-08        40       0.05  -0.050941  -0.102205 -0.012893 -0.017214     -5.106894       4
 2020-07-31 2022-08-08 2022-08-10 2023-02-09     